# Member 4: spectrum_formula.arch2_contrastive, ensemble of trained models (inference only, offline)

Averages the scaled cosines of several saved arch2 models (no training in this notebook), then validates on the team
protocol, applies the library-search gate and writes `submission.csv`.

**Attach before running:** the competition data and your private datasets `casmi-m4-code`, `casmi26-coconut`,
`rdkit-wheel`, `casmi-m4-weights`. Internet off; GPU on.

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
import time

import pandas as pd

T0 = time.time()
QUICK = os.environ.get("M4_QUICK", "0") == "1"      # 1 = tiny wiring check

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input; attach the dataset")
    return hits[0]

CODE = os.path.dirname(find("CASMI_M4_CODE"))
WEIGHTS = os.path.dirname(find("CASMI_M4_WEIGHTS"))      # trained checkpoints (private dataset)
CAND_DIR = "/kaggle/working/candidates"   # candidate tables from separate datasets, gathered in one folder
os.makedirs(CAND_DIR, exist_ok=True)
for name in ("coconut.parquet", "pubchemlite.parquet", "pubchemlite_counts.parquet"):
    try:
        shutil.copy(find(name), CAND_DIR)
    except FileNotFoundError:
        print(f"{name} not attached")
sys.path[:0] = [f"{CODE}/src", f"{CODE}/models/spectrum_formula/arch2_contrastive"]
os.environ.update(CASMI_ENV="kaggle", CASMI_CANDIDATES_DIR=CAND_DIR)
print("code:", CODE, "| candidates:", CAND_DIR, "| python", sys.version.split()[0])

In [ ]:
# RDKit is not in Kaggle's image: install the wheel that matches this Python, offline.
try:
    import rdkit  # noqa: F401
except ImportError:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = find(f"rdkit*{tag}*.whl")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheel], check=True)
    importlib.invalidate_caches()
import rdkit

print("RDKit", rdkit.__version__)   # the metric pins 2026.03.3

In [ ]:
import torch

NAMES = ["c1", "c2", "e20", "pcl"]                       # saved models to average (any subset of these four)
ckpts = [f"{WEIGHTS}/arch2_{n}.pt" for n in NAMES]
cfg = torch.load(ckpts[0], map_location="cpu", weights_only=False)["cfg"]   # settings saved with model 1
cfg["data"].update(data_dir=None, n_val=3000, external_pools=["coconut"])
EXTRA = {}         # per-run data overrides, e.g. {"external_pools": ["coconut", "pubchemlite"]}
RUN_GATE = True    # fuse library search (B2) where it is reliable (Member 7's gate)
cfg["data"].update(EXTRA)
cfg["train"]["device"] = "auto"
cfg["predict"]["ensemble"] = {"checkpoints": ckpts}
cfg["predict"]["prior"] = None
cfg["predict"]["retrieval"] = None   # an ensemble already captures what retrieval adds (+0.001 on top)
EXTRA_PREDICT = {}  # per-run predict overrides, e.g. {"retrieval": {"weight": 20.0, "k": 20, "beta": 30.0}}
cfg["predict"].update(EXTRA_PREDICT)
print(NAMES, "| GPUs:", torch.cuda.device_count(), "| pool:", cfg["data"]["external_pools"])

In [ ]:
# Team protocol: split, pool (+ COCONUT) and pool fingerprints are built once and reused below.
import train as m4_train

proto = m4_train.build_protocol(cfg)
n_test = proto.test.molecule_id.nunique()
print(f"pool {len(proto.pool):,} structures | val molecules {len(proto.truth):,} | test molecules {n_test:,}")

In [ ]:
# Validation on the team protocol, then the test submission (optionally gated with library search).
import predict as m4_predict
from common import b2gate

cp = proto.cp
pool_keys = proto.pool.key.to_numpy()
val_rank, _, truth, _ = m4_predict.predict(cfg, "val", proto)
test_rank, _, _, _ = m4_predict.predict(cfg, "test", proto)
tables = {"model": cp.evaluate(val_rank, truth, pool_keys)}
honest = {"model": float(tables["model"].loc["balanced", "MRR@25"])}
final_rank, chosen, gate_info = test_rank, "model", None
print("model only:\n", tables["model"].to_string())

if RUN_GATE:
    val_g, test_g, gate_info, _ = b2gate.fuse_with_b2(proto, val_rank, test_rank, log=print)
    tables["gated"] = cp.evaluate(val_g, truth, pool_keys)
    honest["gated"] = gate_info["cross_fitted"]          # tau was tuned on validation: judge it out of sample
    print("gate:", gate_info, "\ngated:\n", tables["gated"].to_string())
    if honest["gated"] > honest["model"]:
        final_rank, chosen = test_g, "gated"

cp.write_submission(test_rank, proto.pool, proto.test, path="/kaggle/working/submission_model_only.csv")
sub = cp.write_submission(final_rank, proto.pool, proto.test, path="/kaggle/working/submission.csv")
pd.concat(tables, names=["method"]).to_csv("/kaggle/working/validation_table.csv")
print(f"submission uses: {chosen} | honest val {honest} | {sub.shape} | total {(time.time() - T0) / 60:.1f} min")
sub.head()